### Otimização e Performance

Este notebook tem como objetivo analisar e aplicar técnicas de otimização sobre as tabelas Delta do projeto NYC Taxi.

Serão avaliados aspectos relacionados ao plano de execução das consultas, particionamento dos dados e otimizações disponíveis no Delta Lake, buscando demonstrar os impactos das estratégias utilizadas sobre o processamento distribuído.

**1 - Configuração do ambiente**

In [0]:
from pyspark.sql.functions import *

tabela_silver = "nyc_taxi_data.silver.viagens"

df_silver = spark.table(tabela_silver)

print(f"Quantidade de registros: {df_silver.count():,}")
print(f"Quantidade de colunas: {len(df_silver.columns)}")

Quantidade de registros: 11,198,026
Quantidade de colunas: 42


**2 - Diagnóstico inicial da tabela Silver**

Antes da aplicação de técnicas de otimização, será analisada a estrutura física da tabela Delta utilizada como principal fonte analítica do projeto.

In [0]:
spark.sql("""
DESCRIBE DETAIL nyc_taxi_data.silver.viagens
""").show(truncate=False)

+------+------------------------------------+----------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-------------------------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                        |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                                          |statist

In [0]:
spark.sql("""
DESCRIBE HISTORY nyc_taxi_data.silver.viagens
""").show(truncate=False)

+-------+-------------------+--------------+-------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                       |operation                        |operationParameters                                                                                                    

A análise física da tabela Silver identificou apenas três arquivos Delta para aproximadamente 11,2 milhões de registros, não sendo observado um problema de fragmentação excessiva. Dessa forma, técnicas de compactação de arquivos não foram priorizadas como principal estratégia de otimização.

**3 - Análise do plano de execução**

Será utilizada uma consulta analítica sobre a camada Silver para avaliar o plano de execução gerado pelo Spark antes da aplicação de estratégias adicionais de otimização.

In [0]:
df_analise = (
    df_silver
    .filter(
        (col("Mes_Corrida") == 3) &
        (col("Status_qualidade") == "Valido")
    )
    .groupBy(
        "Distrito_inicio"
    )
    .agg(
        count("*").alias("Qtd_Viagens"),
        sum("Valor_Total_Corrida").alias("Receita_Total"),
        avg("Distancia_corrida_milhas").alias("Distancia_Media")
    )
    .orderBy(
        col("Qtd_Viagens").desc()
    )
)

In [0]:
df_analise.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonSort (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonGroupingAgg (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet nyc_taxi_data.silver.viagens (1)


(1) PhotonScan parquet nyc_taxi_data.silver.viagens
Output [5]: [Distancia_corrida_milhas#12008, Valor_Total_Corrida#12020, Mes_Corrida#12024, Status_qualidade#12039, Distrito_inicio#12040]
DictionaryFilters: [(Status_qualidade#12039 = Valido), (Mes_Corrida#12024 = 3)]
Location: PreparedDeltaFileIndex [s

Análise do plano de execução

O plano físico demonstra que a consulta foi integralmente executada com Photon e utilizou otimizações do Spark/Databricks. A leitura da tabela aplicou seleção apenas das colunas necessárias e filtros sobre Mes_Corrida e Status_qualidade antes das agregações.

Foram identificadas etapas de Shuffle Exchange associadas ao agrupamento por distrito e à ordenação do resultado, operações que exigem redistribuição dos dados entre as partições.

O plano também utiliza AdaptiveSparkPlan, indicando a atuação do Adaptive Query Execution durante o processamento.

In [0]:
import time

In [0]:
inicio = time.time()

resultado = df_analise.collect()

fim = time.time()

tempo_execucao = fim - inicio

print(f"Tempo de execução: {tempo_execucao:.2f} segundos")

Tempo de execução: 2.39 segundos


**Limitação do ambiente Serverless**

Foi avaliado o uso de cache/persistência em memória como estratégia de otimização para consultas repetitivas. Entretanto, o compute Serverless utilizado no Databricks Free Edition não suporta operações de persistência/cache nesse contexto.

Dessa forma, a estratégia não foi aplicada, sendo priorizadas otimizações compatíveis com o ambiente, como análise do plano de execução, Photon, Adaptive Query Execution e redução de leitura por projeção e filtros.

In [0]:
df_analise_sem_order = (
    df_silver
    .filter(
        (col("Mes_Corrida") == 3) &
        (col("Status_qualidade") == "Valido")
    )
    .groupBy("Distrito_inicio")
    .agg(
        count("*").alias("Qtd_Viagens"),
        sum("Valor_Total_Corrida").alias("Receita_Total"),
        avg("Distancia_corrida_milhas").alias("Distancia_Media")
    )
)

In [0]:
df_analise_sem_order.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet nyc_taxi_data.silver.viagens (1)


(1) PhotonScan parquet nyc_taxi_data.silver.viagens
Output [5]: [Distancia_corrida_milhas#12385, Valor_Total_Corrida#12397, Mes_Corrida#12401, Status_qualidade#12416, Distrito_inicio#12417]
DictionaryFilters: [(Mes_Corrida#12401 = 3), (Status_qualidade#12416 = Valido)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-r2yvm/uc/727ecb69-2bb1-457b-93b1-d3373d0481cf/d5be2b54-c42e-4219-ba35-6fd6836333d8/__unitystorage/catalogs/366a7884-f695-4de6-8b2c-4b3059a03845/tables/4925750a-61ac-4c95-b089-6a30011809bd]
ReadSchema: struct<Distancia_corrida_

**Comparação do plano de execução**

A remoção da ordenação global (orderBy) reduziu o plano físico de 14 para 10 etapas. Na consulta original, além do shuffle necessário para o agrupamento por Distrito_inicio, o Spark realizava um segundo shuffle com rangepartitioning, seguido de uma operação de ordenação (PhotonSort).

Na versão sem orderBy, esse segundo shuffle e a etapa de ordenação foram eliminados. O experimento demonstra que operações de ordenação global podem adicionar etapas de redistribuição de dados ao processamento distribuído e devem ser utilizadas apenas quando necessárias ao resultado analítico.

In [0]:
import time

inicio = time.time()

resultado_sem_order = df_analise_sem_order.collect()

fim = time.time()

tempo_sem_order = fim - inicio

print(f"Tempo sem orderBy: {tempo_sem_order:.2f} segundos")

Tempo sem orderBy: 1.17 segundos


**4 - Adaptive Query Execution (AQE) e tratamento de Data Skew**

Será verificado se o Adaptive Query Execution está habilitado no ambiente e se as otimizações relacionadas a skew joins e coalescência dinâmica de partições estão ativas.

O AQE permite que o Spark ajuste o plano de execução em runtime com base nas estatísticas reais observadas durante o processamento.

**Limitação de acesso às configurações AQE**

Foi realizada uma tentativa de consultar diretamente as configurações relacionadas ao Adaptive Query Execution por meio de `spark.conf.get`.

Entretanto, o ambiente Databricks Serverless utilizado no projeto não expõe essas configurações ao usuário, retornando `CONFIG_NOT_AVAILABLE`.

Apesar dessa limitação, o plano físico obtido por `explain("formatted")` apresentou `AdaptiveSparkPlan`, evidenciando a utilização de execução adaptativa pelo mecanismo do Databricks.

In [0]:
df_skew_distrito = (
    df_silver
    .groupBy("Distrito_inicio")
    .agg(
        count("*").alias("Qtd_Viagens")
    )
    .orderBy(
        col("Qtd_Viagens").desc()
    )
)

df_skew_distrito.show(truncate=False)

+---------------+-----------+
|Distrito_inicio|Qtd_Viagens|
+---------------+-----------+
|Manhattan      |9827113    |
|Queens         |965059     |
|Brooklyn       |306011     |
|Bronx          |70406      |
|Unknown        |23298      |
|N/A            |4093       |
|EWR            |1056       |
|Staten Island  |990        |
+---------------+-----------+



**Análise do Data Skew**

A análise da distribuição das viagens por distrito de origem identificou forte assimetria nos dados.

Manhattan concentra aproximadamente 9,8 milhões dos 11,2 milhões de registros da camada Silver, correspondendo a cerca de 88% das viagens. Queens representa aproximadamente 8,6%, enquanto os demais distritos possuem participação significativamente menor.

Essa concentração caracteriza um cenário de **data skew**, no qual determinadas chaves possuem volume de dados muito superior às demais. Em operações distribuídas baseadas nessas chaves, como agrupamentos e joins, essa assimetria pode gerar partições com cargas de processamento desiguais.

O plano físico analisado anteriormente apresentou `AdaptiveSparkPlan`, evidenciando a utilização de execução adaptativa pelo ambiente. Entretanto, por se tratar de Databricks Serverless, as configurações internas específicas relacionadas ao AQE e ao tratamento de skew não estão disponíveis para consulta direta por meio de `spark.conf.get`.

###Resultado do teste de otimização

A consulta original, contendo orderBy, apresentou tempo de execução de 2,39 segundos. Após a remoção da ordenação global, o tempo observado foi de 1,17 segundos, representando uma redução aproximada de 51%.

A análise dos planos físicos mostrou que a versão original possuía dois processos de shuffle: um associado ao groupBy e outro à ordenação global. Na versão otimizada, permaneceram apenas as etapas de redistribuição necessárias para a agregação, sendo eliminados o segundo Shuffle Exchange e o PhotonSort.

O teste demonstra o impacto que operações de ordenação global podem exercer sobre o processamento distribuído. Quando a ordenação não for necessária para o resultado analítico, sua remoção pode simplificar o plano de execução e reduzir o tempo de processamento.

Observação: os tempos medidos representam uma execução específica no ambiente Serverless e podem variar entre execuções. A principal evidência estrutural da otimização é a simplificação observada no plano físico.